# Notebook 02  
# PPO-Based Adaptive Stabilization of a Nonlinear 2-DOF Robotic Manipulator

---

## Purpose

This notebook implements and trains a Proximal Policy Optimization (PPO)
agent to stabilize a nonlinear two-degree-of-freedom (2-DOF) robotic
manipulator around an upright configuration.

Unlike classical PD control, PPO learns a control policy
directly from interaction with the system dynamics.

---

## Scope

This notebook covers:

1. Environment formulation
2. Reward function design
3. PPO configuration
4. Training process
5. Policy evaluation
6. Preliminary performance observations

Robustness and comparative analysis will be performed in later notebooks.

## 1. Problem Formulation

We consider a nonlinear two-degree-of-freedom (2-DOF) robotic manipulator
simulated in MuJoCo.

The system evolves according to nonlinear rigid-body dynamics.
Without control input, the manipulator is unstable under gravity.

### Control Objective

Stabilize the manipulator around the upright configuration:

- Joint 1 target: π radians
- Joint 2 target: 0 radians

### Learning Objective

Train a reinforcement learning agent using Proximal Policy Optimization (PPO)
to learn a torque control policy that:

- Minimizes position error
- Reduces joint velocity
- Avoids excessive torque usage

Unlike classical control, the agent does not explicitly use the system model.
It learns control behavior purely through interaction.

## 2. Environment Design

To apply reinforcement learning, the robotic manipulator
must be wrapped as a Gym-compatible environment.

The environment defines how the agent interacts with the system.

### Observation Space

The state vector consists of:

- Joint positions: (q1, q2)
- Joint velocities: (q1_dot, q2_dot)

Total state dimension: 4

### Action Space

The action represents the torque applied at each joint:

- Torque 1 (tau1)
- Torque 2 (tau2)

The action space is continuous.

### Episode Design

Each episode:

- Starts near the upright configuration
- Runs for a fixed number of simulation steps
- Does not terminate early for now

The agent must learn to keep the system near the target state.


In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import mujoco


class ArmStabilizationEnv(gym.Env):

    def __init__(self, model_path):
        super().__init__()

        self.model = mujoco.MjModel.from_xml_path(model_path)
        self.data = mujoco.MjData(self.model)

        self.q_des = np.array([np.pi, 0.0])

        high = np.array([np.inf, np.inf, np.inf, np.inf], dtype=np.float32)
        self.observation_space = spaces.Box(-high, high, dtype=np.float32)

        self.action_space = spaces.Box(
            low=np.array([-500.0, -500.0], dtype=np.float32),
            high=np.array([500.0, 500.0], dtype=np.float32),
            dtype=np.float32
        )

        self.max_steps = 1000
        self.current_step = 0

    def _get_obs(self):
        return np.concatenate([self.data.qpos, self.data.qvel]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.data.qpos[:] = self.q_des + np.random.uniform(-0.1, 0.1, size=2)
        self.data.qvel[:] = np.random.uniform(-0.1, 0.1, size=2)

        self.current_step = 0
        return self._get_obs(), {}

    def step(self, action):
        self.current_step += 1

        action = np.clip(action, self.action_space.low, self.action_space.high)
        self.data.ctrl[:] = action
        mujoco.mj_step(self.model, self.data)

        obs = self._get_obs()

        pos_error = self.data.qpos - self.q_des
        vel = self.data.qvel

        reward = (
            -10.0 * np.sum(pos_error**2)
            -1.0 * np.sum(vel**2)
            -0.0001 * np.sum(action**2)
        )

        terminated = False
        truncated = self.current_step >= self.max_steps

        return obs, reward, terminated, truncated, {}